In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import poisson

files = sorted(Path("../data/raw").glob("seriea_*.csv"))
df = pd.concat([pd.read_csv(f).assign(Season=f.stem[-4:]) for f in files], ignore_index=True)
df = df[["Season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]].dropna()
df[["FTHG", "FTAG"]] = df[["FTHG", "FTAG"]].astype(int)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed")
df = df.sort_values("Date").reset_index(drop=True)

C:\Users\Utente\AppData\Local\Temp\ipykernel_1972\1612280969.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = pd.concat([pd.read_csv(f).assign(Season=f.stem[-4:]) for f in files], ignore_index=True)


In [2]:
media = df["FTHG"].mean()
reale = df["FTHG"].value_counts(normalize=True).sort_index()
pd.DataFrame({"reale": reale, "poisson": poisson.pmf(reale.index, media)}).round(3)

,reale,poisson
FTHG,,
0,0.235,0.227
1,0.332,0.336
2,0.245,0.250
3,0.120,0.124
4,0.046,0.046
5,0.016,0.014
6,0.005,0.003
7,0.001,0.001
8,0.000,0.000


In [3]:
def stima_forze(train):
    casa = train.rename(columns={"HomeTeam": "team", "FTHG": "gf", "FTAG": "gs"})
    trasf = train.rename(columns={"AwayTeam": "team", "FTAG": "gf", "FTHG": "gs"})
    tutte = pd.concat([casa, trasf])[["team", "gf", "gs"]]
    medie = tutte.groupby("team").mean()
    media_gol = tutte["gf"].mean()
    forze = pd.DataFrame({"att": medie["gf"] / media_gol, "dif": medie["gs"] / media_gol})
    return forze, train["FTHG"].mean(), train["FTAG"].mean()

In [4]:
NEOPROMOSSA = (1.0, 1.0)

def prob_partita(casa, trasf, forze, media_casa, media_trasf, max_gol=10):
    att_c, dif_c = forze.loc[casa] if casa in forze.index else NEOPROMOSSA
    att_t, dif_t = forze.loc[trasf] if trasf in forze.index else NEOPROMOSSA
    lam = media_casa * att_c * dif_t
    mu = media_trasf * att_t * dif_c
    gol = np.arange(max_gol + 1)
    matrice = np.outer(poisson.pmf(gol, lam), poisson.pmf(gol, mu))
    p_casa = np.tril(matrice, -1).sum()
    p_pari = np.trace(matrice)
    p_trasf = np.triu(matrice, 1).sum()
    return p_casa, p_pari, p_trasf, matrice

In [5]:
train = df[df["Season"].isin(["2425", "2526", "2627"])]
forze, mc, mt = stima_forze(train)
print(forze.sort_values("att", ascending=False).round(2).head(8))

p1, px, p2, m = prob_partita("Inter", "Milan", forze, mc, mt)
print(f"\nInter-Milan → 1: {p1:.1%}   X: {px:.1%}   2: {p2:.1%}")
pd.DataFrame(m[:5, :5]).round(3)

            att   dif
team                 
Inter      1.75  0.74
Frosinone  1.70  1.06
Roma       1.31  0.68
Atalanta   1.31  0.83
Milan      1.22  0.81
Napoli     1.21  0.69
Juventus   1.20  0.72
Como       1.18  0.81

Inter-Milan → 1: 56.1%   X: 22.1%   2: 21.8%


,0,1,2,3,4
0,0.048,0.054,0.030,0.011,0.003
1,0.092,0.103,0.057,0.021,0.006
2,0.089,0.099,0.055,0.020,0.006
3,0.057,0.063,0.035,0.013,0.004
4,0.027,0.030,0.017,0.006,0.002


In [6]:
stagioni = sorted(df["Season"].unique())
righe = []
for i, s in enumerate(stagioni):
    if i < 3:
        continue
    train = df[df["Season"].isin(stagioni[i-3:i])]
    forze, mc, mt = stima_forze(train)
    for row in df[df["Season"] == s].itertuples():
        p1, px, p2, _ = prob_partita(row.HomeTeam, row.AwayTeam, forze, mc, mt)
        righe.append({"idx": row.Index, "pH": p1, "pD": px, "pA": p2})

ris = df.join(pd.DataFrame(righe).set_index("idx"), how="inner")

In [7]:
test = ris[ris["Date"] >= "2023-07-01"]
probs = test[["pH", "pD", "pA"]].to_numpy()
probs = probs / probs.sum(axis=1, keepdims=True)

pred = np.array(["H", "D", "A"])[probs.argmax(axis=1)]
acc = (pred == test["FTR"]).mean()

col = test["FTR"].map({"H": 0, "D": 1, "A": 2}).to_numpy()
logloss = -np.log(probs[np.arange(len(test)), col]).mean()

freq = test["FTR"].value_counts(normalize=True)
logloss_base = -np.log(test["FTR"].map(freq)).mean()

print(f"Accuratezza Poisson: {acc:.1%}   (Elo: 53,0%)")
print(f"Log-loss Poisson: {logloss:.3f}   Baseline: {logloss_base:.3f}")
print("Pareggi previsti:", (pred == "D").sum(), "su", len(test))

Accuratezza Poisson: 51.5%   (Elo: 53,0%)
Log-loss Poisson: 0.993   Baseline: 1.084
Pareggi previsti: 0 su 1250
